Full Final Attempt

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.preprocessing import StandardScaler
import numpy as np
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

def process_avg_teamstats_data(tourney_filename, season_filename):
    # Load datasets
    tourney_results = pd.read_csv(os.path.join(data_path, tourney_filename))
    season_results = pd.read_csv(os.path.join(data_path, season_filename))

    # Calculate advanced stats separately for winners and losers
    def calculate_advanced_stats(df, team_prefix):
        stats = pd.DataFrame({
            'Season': df['Season'],
            'TeamID': df[f'{team_prefix}TeamID'],
            'Score': df[f'{team_prefix}Score'],
            'FGM': df[f'{team_prefix}FGM'],
            'FGA': df[f'{team_prefix}FGA'],
            'FGM3': df[f'{team_prefix}FGM3'],
            'FGA3': df[f'{team_prefix}FGA3'],
            'FTM': df[f'{team_prefix}FTM'],
            'FTA': df[f'{team_prefix}FTA'],
            'OR': df[f'{team_prefix}OR'],
            'DR': df[f'{team_prefix}DR'],
            'Ast': df[f'{team_prefix}Ast'],
            'TO': df[f'{team_prefix}TO']
        })
        stats["FG%"] = stats["FGM"] / stats["FGA"]
        stats["3P%"] = stats["FGM3"] / stats["FGA3"]
        stats["FT%"] = stats["FTM"] / stats["FTA"]
        stats["OR%"] = stats["OR"] / (stats["OR"] + stats["DR"])
        stats["DR%"] = stats["DR"] / (stats["OR"] + stats["DR"])
        stats["AST/TO"] = stats["Ast"] / stats["TO"]
        stats["NetRating"] = 100 * stats["Score"] / (stats["FGA"] + 0.44 * stats["FTA"] + stats["TO"])
        return stats

    # Compute for winners and losers separately
    winners_stats = calculate_advanced_stats(season_results, 'W')
    losers_stats = calculate_advanced_stats(season_results, 'L')

    # Combine all game data for each team
    all_team_stats = pd.concat([winners_stats, losers_stats])

    # Aggregate all season performance
    team_stats = all_team_stats.groupby(["Season", "TeamID"]).agg(
        GamesPlayed=("TeamID", "count"),
        Wins=("Score", lambda x: (x >= x.mean()).sum()),  # approximate wins
        AvgScore=("Score", "mean"),
        AvgFGP=("FG%", "mean"),
        Avg3PP=("3P%", "mean"),
        AvgFT=("FT%", "mean"),
        AvgORP=("OR%", "mean"),
        AvgDRP=("DR%", "mean"),
        AvgASTTO=("AST/TO", "mean"),
        AvgNetRtg=("NetRating", "mean"),
    ).reset_index()

    return team_stats

# Define the data path
data_path = "."

# Processing Clearly for Balanced Stats:
men_teamstats = process_avg_teamstats_data("data\MNCAATourneyDetailedResults.csv", "data\MRegularSeasonDetailedResults.csv")
women_teamstats = process_avg_teamstats_data("data\WNCAATourneyDetailedResults.csv", "data\WRegularSeasonDetailedResults.csv")

# Save the processed data to CSV files
men_teamstats.to_csv("data\Processed_Men_TeamStats.csv", index=False)
women_teamstats.to_csv("data\Processed_Women_TeamStats.csv", index=False)

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import make_scorer
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def load_and_merge_data(team_stats_path, regular_season_path, tourney_path):
    team_stats = pd.read_csv(team_stats_path)
    regular_season_results = pd.read_csv(regular_season_path)
    tourney_results = pd.read_csv(tourney_path)

    # Combine regular season and tournament results
    results = pd.concat([regular_season_results, tourney_results])

    # Merge team stats for winners and losers separately
    results = results.merge(team_stats, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'])
    results = results.merge(team_stats, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'], suffixes=('', '_L'))

    # Drop the redundant 'TeamID' columns if they exist
    results.drop(columns=[col for col in ['TeamID_x', 'TeamID_y'] if col in results.columns], inplace=True)

    return results

# Merge datasets and save clearly:
men_team_stats_path = 'data\Processed_Men_TeamStats.csv'
men_regular_season_path = 'data\MRegularSeasonCompactResults.csv'
men_tourney_path = 'data\MNCAATourneyCompactResults.csv'

women_team_stats_path = 'data\Processed_Women_TeamStats.csv'
women_regular_season_path = 'data\WRegularSeasonCompactResults.csv'
women_tourney_path = 'data\WNCAATourneyCompactResults.csv'

men_merged = load_and_merge_data(men_team_stats_path, men_regular_season_path, men_tourney_path)
women_merged = load_and_merge_data(women_team_stats_path, women_regular_season_path, women_tourney_path)

final_merged = pd.concat([men_merged, women_merged])
final_merged.to_csv('final_merged.csv', index=False)

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Load the data
df = pd.read_csv('final_merged.csv')

# Define columns to drop (metadata)
drop_cols = ['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WScore', 'LScore', 'WLoc', 'NumOT']
features = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
            'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg',
            'GamesPlayed_L', 'Wins_L', 'AvgScore_L', 'AvgFGP_L', 'Avg3PP_L', 'AvgFT_L',
            'AvgORP_L', 'AvgDRP_L', 'AvgASTTO_L', 'AvgNetRtg_L']

# Initial dataset (Team1 wins)
X = df[features]
y = np.ones(len(df), dtype=int)

# Invert data (Team2 wins)
df_inv = df.copy()
for col in features:
    if col.endswith('_L'):
        base_col = col[:-2]
        df_inv[col], df_inv[base_col] = df[base_col], df[col]

X_inv = df_inv[features]
y_inv = np.zeros(len(df_inv), dtype=int)

# Combine both datasets
X_train_full = pd.concat([X, X_inv], ignore_index=True)
y_train_full = np.concatenate([y, y_inv])

# ✅ FIX: Explicitly Replace infinite values
X_train_full.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check for NaNs or infinite values explicitly again
print("Infinite values present after replacement:", np.isinf(X_train_full).sum().sum())
print("NaNs before imputation:", X_train_full.isna().sum().sum())

# Impute missing values clearly
imputer = SimpleImputer(strategy='median')
X_train_full = pd.DataFrame(imputer.fit_transform(X_train_full), columns=features)

# Verify no missing or infinite values remain
assert not np.isinf(X_train_full.values).any(), "Infinite values remain!"
assert not np.isnan(X_train_full.values).any(), "NaN values remain after imputation."

# Train Random Forest clearly
rf_model = RandomForestClassifier(n_estimators=200, max_depth=7, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_model.fit(X_train_full, y_train_full)

print("✅ Model trained successfully without infinities or NaNs.")

# Load submission_df
submission_df = pd.read_csv('data/SampleSubmissionStage2.csv')
submission_df[['Season', 'LowerID', 'HigherID']] = submission_df['ID'].str.split('_', expand=True).astype(int)

# Define team stat columns explicitly
team_stats_cols = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
                   'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg']

# Create a clear lookup DataFrame for team stats
winner_stats_df = df.set_index(['Season', 'WTeamID'])[team_stats_cols]
loser_stats_df = df.set_index(['Season', 'LTeamID'])[[col + '_L' for col in team_stats_cols]]
loser_stats_cols = {col + '_L': col for col in team_stats_cols}
loser_stats_df.rename(columns=loser_stats_cols, inplace=True)

# Combine winner and loser stats into one lookup DataFrame
all_team_stats_df = pd.concat([winner_stats_df, loser_stats_df])
all_team_stats_df = all_team_stats_df[~all_team_stats_df.index.duplicated(keep='first')]  # Remove duplicates clearly

# Function to safely get stats
def get_team_stats(season, team_id):
    if (season, team_id) in all_team_stats_df.index:
        stats = all_team_stats_df.loc[(season, team_id)]
        # ensure it returns a list from Series
        if isinstance(stats, pd.Series):
            return stats.tolist()
        else:  # If it's a DataFrame due to multiple matches, take the first row
            return stats.iloc[0].tolist()
    else:
        return X_train_full[team_stats_cols].median().tolist()

# Construct test set efficiently
X_test_rows = [
    get_team_stats(season, lower) + get_team_stats(season, higher)
    for season, lower, higher in submission_df[['Season', 'LowerID', 'HigherID']].itertuples(index=False)
]

X_test = pd.DataFrame(X_test_rows, columns=[f'{col}_1' for col in team_stats_cols] + [f'{col}_2' for col in team_stats_cols])

# Check the feature alignment clearly
print("✅ Features prepared for testing.")
print("Features used by model:", rf_model.feature_names_in_)
print("Test DataFrame columns:", X_test.columns.tolist())

missing_in_test = set(rf_model.feature_names_in_) - set(X_test.columns)
extra_in_test = set(X_test.columns) - set(rf_model.feature_names_in_)

print("Missing in test (should be empty):", missing_in_test)
print("Extra in test (should be empty):", extra_in_test)

# Confirm dimensions
print(X_test.shape) # Should clearly show (131407, 20)

# Check initial infinite values and NaNs explicitly
print("Infinite values before replacement:", np.isinf(X_test.values).sum())
print("NaNs before replacement:", X_test.isna().sum().sum())

# Replace infinite values explicitly with NaN
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

# Verify intermediate NaNs after replacement
print("NaNs after replacing infinities:", X_test.isna().sum().sum())

# Explicitly rename columns BEFORE imputation to align with training data
rename_mapping = {
    'GamesPlayed_1': 'GamesPlayed',
    'Wins_1': 'Wins',
    'AvgScore_1': 'AvgScore',
    'AvgFGP_1': 'AvgFGP',
    'Avg3PP_1': 'Avg3PP',
    'AvgFT_1': 'AvgFT',
    'AvgORP_1': 'AvgORP',
    'AvgDRP_1': 'AvgDRP',
    'AvgASTTO_1': 'AvgASTTO',
    'AvgNetRtg_1': 'AvgNetRtg',
    'GamesPlayed_2': 'GamesPlayed_L',
    'Wins_2': 'Wins_L',
    'AvgScore_2': 'AvgScore_L',
    'AvgFGP_2': 'AvgFGP_L',
    'Avg3PP_2': 'Avg3PP_L',
    'AvgFT_2': 'AvgFT_L',
    'AvgORP_2': 'AvgORP_L',
    'AvgDRP_2': 'AvgDRP_L',
    'AvgASTTO_2': 'AvgASTTO_L',
    'AvgNetRtg_2': 'AvgNetRtg_L'
}
X_test.rename(columns=rename_mapping, inplace=True)

# Impute missing values explicitly using aligned columns with training median
for col in X_test.columns:
    median_val = X_train_full[col].median()
    X_test[col].fillna(median_val, inplace=True)

# Recheck explicitly to confirm no NaNs or infinities remain
assert np.isinf(X_test.values).sum() == 0, "Still contains infinities!"
assert X_test.isna().sum().sum() == 0, "Still contains NaNs!"

# Ensure data type consistency explicitly
X_test = X_test.astype(np.float64)

# Confirm feature alignment explicitly
assert set(X_test.columns) == set(rf_model.feature_names_in_), \
    f"Mismatch found: {set(rf_model.feature_names_in_) - set(X_test.columns)}"

# Generate predictions explicitly matching the model feature order
submission_df['Pred'] = rf_model.predict_proba(X_test[rf_model.feature_names_in_])[:, 1]

# Save predictions to CSV explicitly
submission_df[['ID', 'Pred']].to_csv('final_submission.csv', index=False)
print(submission_df[['ID', 'Pred']].shape)
print("✅ Final submission generated successfully: final_submission.csv")

Infinite values present after replacement: 0
NaNs before imputation: 174
✅ Model trained successfully without infinities or NaNs.
✅ Features prepared for testing.
Features used by model: ['GamesPlayed' 'Wins' 'AvgScore' 'AvgFGP' 'Avg3PP' 'AvgFT' 'AvgORP'
 'AvgDRP' 'AvgASTTO' 'AvgNetRtg' 'GamesPlayed_L' 'Wins_L' 'AvgScore_L'
 'AvgFGP_L' 'Avg3PP_L' 'AvgFT_L' 'AvgORP_L' 'AvgDRP_L' 'AvgASTTO_L'
 'AvgNetRtg_L']
Test DataFrame columns: ['GamesPlayed_1', 'Wins_1', 'AvgScore_1', 'AvgFGP_1', 'Avg3PP_1', 'AvgFT_1', 'AvgORP_1', 'AvgDRP_1', 'AvgASTTO_1', 'AvgNetRtg_1', 'GamesPlayed_2', 'Wins_2', 'AvgScore_2', 'AvgFGP_2', 'Avg3PP_2', 'AvgFT_2', 'AvgORP_2', 'AvgDRP_2', 'AvgASTTO_2', 'AvgNetRtg_2']
Missing in test (should be empty): {'Wins', 'AvgDRP', 'AvgFT', 'AvgORP_L', 'Avg3PP_L', 'AvgNetRtg', 'Wins_L', 'AvgFGP', 'AvgFGP_L', 'Avg3PP', 'AvgScore', 'AvgFT_L', 'AvgASTTO_L', 'AvgORP', 'GamesPlayed', 'AvgASTTO', 'GamesPlayed_L', 'AvgDRP_L', 'AvgScore_L', 'AvgNetRtg_L'}
Extra in test (should be empty): 

Attempt failure

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, brier_score_loss
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

# -------------------- STEP 1: DATA PREPROCESSING --------------------

# Function to calculate advanced basketball statistics
def calculate_advanced_stats(df, prefix):
    stats = pd.DataFrame({
        'Season': df['Season'],
        'TeamID': df[f'{prefix}TeamID'],
        'Score': df[f'{prefix}Score'],
        'FG%': df[f'{prefix}FGM'] / df[f'{prefix}FGA'],
        '3P%': df[f'{prefix}FGM3'] / df[f'{prefix}FGA3'],
        'FT%': df[f'{prefix}FTM'] / df[f'{prefix}FTA'],
        'OR%': df[f'{prefix}OR'] / (df[f'{prefix}OR'] + df[f'{prefix}DR']),
        'DR%': df[f'{prefix}DR'] / (df[f'{prefix}OR'] + df[f'{prefix}DR']),
        'AST/TO': df[f'{prefix}Ast'] / df[f'{prefix}TO'],
        'NetRating': 100 * df[f'{prefix}Score'] / (df[f'{prefix}FGA'] + 0.44 * df[f'{prefix}FTA'] + df[f'{prefix}TO'])
    })
    return stats

# Aggregate season statistics for each team
def process_season_team_stats(tourney_file, season_file):
    tourney_results = pd.read_csv(tourney_file)
    season_results = pd.read_csv(season_file)

    winners_stats = calculate_advanced_stats(season_results, 'W')
    losers_stats = calculate_advanced_stats(season_results, 'L')

    all_stats = pd.concat([winners_stats, losers_stats])

    aggregated_stats = all_stats.groupby(['Season', 'TeamID']).agg(
        GamesPlayed=('TeamID', 'size'),
        Wins=('Score', lambda x: (x >= x.mean()).sum()),
        AvgScore=('Score', 'mean'),
        AvgFGP=('FG%', 'mean'),
        Avg3PP=('3P%', 'mean'),
        AvgFT=('FT%', 'mean'),
        AvgORP=('OR%', 'mean'),
        AvgDRP=('DR%', 'mean'),
        AvgASTTO=('AST/TO', 'mean'),
        AvgNetRtg=('NetRating', 'mean')
    ).reset_index()

    return aggregated_stats

# -------------------- STEP 2: MERGE & COMBINE DATA --------------------

def merge_results_with_stats(team_stats_file, season_results_file, tourney_results_file):
    team_stats = pd.read_csv(team_stats_file)
    season_results = pd.read_csv(season_results_file)
    tourney_results = pd.read_csv(tourney_results_file)

    combined_results = pd.concat([season_results, tourney_results])

    merged_df = combined_results.merge(team_stats, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'])
    merged_df = merged_df.merge(team_stats, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'], suffixes=('', '_L'))
    merged_df.drop(['TeamID', 'TeamID_L'], axis=1, inplace=True)

    return merged_df

# -------------------- STEP 3: BALANCE TRAINING DATA --------------------

def prepare_training_data(df):
    features = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
                'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg',
                'GamesPlayed_L', 'Wins_L', 'AvgScore_L', 'AvgFGP_L', 'Avg3PP_L', 'AvgFT_L',
                'AvgORP_L', 'AvgDRP_L', 'AvgASTTO_L', 'AvgNetRtg_L']

    X_win = df[features]
    y_win = np.ones(len(df))

    df_inv = df.copy()
    for col in features:
        if '_L' in col:
            base_col = col[:-2]
            df_inv[col], df_inv[base_col] = df[base_col], df[col]

    X_loss = df_inv[features]
    y_loss = np.zeros(len(df_inv))

    X_full = pd.concat([X_win, X_loss], ignore_index=True)
    y_full = np.concatenate([y_win, y_loss])

    X_full.replace([np.inf, -np.inf], np.nan, inplace=True)
    imputer = SimpleImputer(strategy='median')
    X_full_imputed = pd.DataFrame(imputer.fit_transform(X_full), columns=features)

    return X_full_imputed, y_full

# -------------------- STEP 4: MODEL TRAINING --------------------

def train_model(X, y):
    rf_model = RandomForestClassifier(n_estimators=200, max_depth=7, min_samples_leaf=3, random_state=42)
    rf_model.fit(X, y)
    return rf_model

# -------------------- STEP 5: GENERATE PREDICTIONS --------------------

def create_test_features(submission_df, stats_lookup, median_values):
    test_rows = []
    for _, row in submission_df.iterrows():
        season, lower, higher = row['Season'], row['LowerID'], row['HigherID']
        lower_stats = stats_lookup.get((season, lower), median_values).tolist()
        higher_stats = stats_lookup.get((season, higher), median_values).tolist()
        test_rows.append(lower_stats + higher_stats)

    return pd.DataFrame(test_rows, columns=median_values.index.tolist() * 2)

# -------------------- STEP 6: RUN THE ENTIRE PIPELINE --------------------

def run_pipeline():
    final_merged = pd.read_csv('final_merged.csv')
    X, y = prepare_training_data(final_merged)
    model = train_model(X, y)

    submission_df = pd.read_csv('data/SampleSubmissionStage2.csv')
    submission_df[['Season', 'LowerID', 'HigherID']] = submission_df['ID'].str.split('_', expand=True).astype(int)

    team_stats_cols = X.columns[:10]
    stats_lookup = {(row.Season, row.WTeamID): row[team_stats_cols].tolist() for _, row in final_merged.iterrows()}
    median_values = X.median()

    X_test = create_test_features(submission_df, stats_lookup, median_values)

    submission_df['Pred'] = model.predict_proba(X_test)[:, 1]
    submission_df[['ID', 'Pred']].to_csv('final_submission1.csv', index=False)

    print("✅ Pipeline executed successfully. Submission file saved.")

# Execute pipeline
run_pipeline()

SyntaxError: invalid syntax (3211985257.py, line 107)

Attempt 2

In [ ]:
# ✅ Full Corrected NCAA Tournament Prediction Pipeline

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
import os

# Silence warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Paths
DATA_PATH = 'data/'

# Define columns clearly
team_stats_cols = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
                   'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg']
features = team_stats_cols + [col + '_L' for col in team_stats_cols]

# 1️⃣ Load merged dataset clearly
final_merged = pd.read_csv('final_merged.csv')

# 2️⃣ Create balanced training data explicitly
X_win = final_merged[features]
y_win = np.ones(len(final_merged), dtype=int)

# Reverse winner-loser roles clearly
final_merged_inv = final_merged.copy()
for col in team_stats_cols:
    final_merged_inv[col], final_merged_inv[col + '_L'] = final_merged[col + '_L'], final_merged[col]

X_loss = final_merged_inv[features]
y_loss = np.zeros(len(final_merged_inv), dtype=int)

X_train_full = pd.concat([X_win, X_loss], ignore_index=True)
y_train_full = np.concatenate([y_win, y_loss])

# 3️⃣ Handle infinities and NaNs explicitly
X_train_full.replace([np.inf, -np.inf], np.nan, inplace=True)

imputer = SimpleImputer(strategy='median')
X_train_full = pd.DataFrame(imputer.fit_transform(X_train_full), columns=features)

# 4️⃣ Train Random Forest Classifier
model = RandomForestClassifier(n_estimators=200, max_depth=7, min_samples_leaf=3,
                               random_state=42, n_jobs=-1)
model.fit(X_train_full, y_train_full)

# 5️⃣ Prepare submission data explicitly
submission_df = pd.read_csv(os.path.join(DATA_PATH, 'SampleSubmissionStage2.csv'))
submission_df[['Season', 'LowerID', 'HigherID']] = submission_df['ID'].str.split('_', expand=True).astype(int)

# Create lookup dictionary efficiently
stats_lookup = {}
for _, row in final_merged.iterrows():
    stats_lookup[(row['Season'], row['WTeamID'])] = row[team_stats_cols].tolist()
    stats_lookup[(row['Season'], row['LTeamID'])] = row[[col + '_L' for col in team_stats_cols]].tolist()

# Define median values explicitly
median_lower = X_train_full[team_stats_cols].median().tolist()
median_higher = X_train_full[[col + '_L' for col in team_stats_cols]].median().tolist()

# Function clearly building test features
def create_test_features(sub_df, lookup, med_lower, med_higher):
    test_rows = []
    for season, lower, higher in sub_df[['Season', 'LowerID', 'HigherID']].itertuples(index=False):
        lower_stats = lookup.get((season, lower), med_lower)
        higher_stats = lookup.get((season, higher), med_higher)
        test_rows.append(lower_stats + higher_stats)
    
    return pd.DataFrame(test_rows, columns=features)

X_test = create_test_features(submission_df, stats_lookup, median_lower, median_higher)

# 6️⃣ Explicitly handle infinite values and NaNs in test set
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test = pd.DataFrame(imputer.transform(X_test), columns=features)

# 7️⃣ Predict and save results
submission_df['Pred'] = model.predict_proba(X_test)[:, 1]
submission_df[['ID', 'Pred']].to_csv('final_submission2.csv', index=False)

print("✅ Final submission generated successfully: final_submission.csv")


✅ Final submission generated successfully: final_submission.csv


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.preprocessing import StandardScaler
import numpy as np
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def process_avg_teamstats_data(tourney_filename, season_filename):
    # Load datasets
    tourney_results = pd.read_csv(os.path.join(data_path, tourney_filename))
    season_results = pd.read_csv(os.path.join(data_path, season_filename))

    # Calculate advanced stats separately for winners and losers
    def calculate_advanced_stats(df, team_prefix):
        stats = pd.DataFrame({
            'Season': df['Season'],
            'TeamID': df[f'{team_prefix}TeamID'],
            'Score': df[f'{team_prefix}Score'],
            'FGM': df[f'{team_prefix}FGM'],
            'FGA': df[f'{team_prefix}FGA'],
            'FGM3': df[f'{team_prefix}FGM3'],
            'FGA3': df[f'{team_prefix}FGA3'],
            'FTM': df[f'{team_prefix}FTM'],
            'FTA': df[f'{team_prefix}FTA'],
            'OR': df[f'{team_prefix}OR'],
            'DR': df[f'{team_prefix}DR'],
            'Ast': df[f'{team_prefix}Ast'],
            'TO': df[f'{team_prefix}TO']
        })
        stats["FG%"] = stats["FGM"] / stats["FGA"]
        stats["3P%"] = stats["FGM3"] / stats["FGA3"]
        stats["FT%"] = stats["FTM"] / stats["FTA"]
        stats["OR%"] = stats["OR"] / (stats["OR"] + stats["DR"])
        stats["DR%"] = stats["DR"] / (stats["OR"] + stats["DR"])
        stats["AST/TO"] = stats["Ast"] / stats["TO"]
        stats["NetRating"] = 100 * stats["Score"] / (stats["FGA"] + 0.44 * stats["FTA"] + stats["TO"])
        return stats

    # Compute for winners and losers separately
    winners_stats = calculate_advanced_stats(season_results, 'W')
    losers_stats = calculate_advanced_stats(season_results, 'L')

    # Combine all game data for each team
    all_team_stats = pd.concat([winners_stats, losers_stats])

    # Aggregate all season performance
    team_stats = all_team_stats.groupby(["Season", "TeamID"]).agg(
        GamesPlayed=("TeamID", "count"),
        Wins=("Score", lambda x: (x >= x.mean()).sum()),  # approximate wins
        AvgScore=("Score", "mean"),
        AvgFGP=("FG%", "mean"),
        Avg3PP=("3P%", "mean"),
        AvgFT=("FT%", "mean"),
        AvgORP=("OR%", "mean"),
        AvgDRP=("DR%", "mean"),
        AvgASTTO=("AST/TO", "mean"),
        AvgNetRtg=("NetRating", "mean"),
    ).reset_index()

    return team_stats


In [3]:
# Define the data path
data_path = "."

# Processing Clearly for Balanced Stats:
men_teamstats = process_avg_teamstats_data("data\MNCAATourneyDetailedResults.csv", "data\MRegularSeasonDetailedResults.csv")
women_teamstats = process_avg_teamstats_data("data\WNCAATourneyDetailedResults.csv", "data\WRegularSeasonDetailedResults.csv")

# Save the processed data to CSV files
men_teamstats.to_csv("data\Processed_Men_TeamStats.csv", index=False)
women_teamstats.to_csv("data\Processed_Women_TeamStats.csv", index=False)


In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import make_scorer
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def load_and_merge_data(team_stats_path, regular_season_path, tourney_path):
    team_stats = pd.read_csv(team_stats_path)
    regular_season_results = pd.read_csv(regular_season_path)
    tourney_results = pd.read_csv(tourney_path)

    # Combine regular season and tournament results
    results = pd.concat([regular_season_results, tourney_results])

    # Merge team stats for winners and losers separately
    results = results.merge(team_stats, left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID'])
    results = results.merge(team_stats, left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID'], suffixes=('', '_L'))

    # Drop the redundant 'TeamID' columns if they exist
    results.drop(columns=[col for col in ['TeamID_x', 'TeamID_y'] if col in results.columns], inplace=True)

    return results

# Merge datasets and save clearly:
men_team_stats_path = 'data\Processed_Men_TeamStats.csv'
men_regular_season_path = 'data\MRegularSeasonCompactResults.csv'
men_tourney_path = 'data\MNCAATourneyCompactResults.csv'

women_team_stats_path = 'data\Processed_Women_TeamStats.csv'
women_regular_season_path = 'data\WRegularSeasonCompactResults.csv'
women_tourney_path = 'data\WNCAATourneyCompactResults.csv'

men_merged = load_and_merge_data(men_team_stats_path, men_regular_season_path, men_tourney_path)
women_merged = load_and_merge_data(women_team_stats_path, women_regular_season_path, women_tourney_path)

final_merged = pd.concat([men_merged, women_merged])
final_merged.to_csv('final_merged.csv', index=False)

In [5]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# Load the data
df = pd.read_csv('final_merged.csv')

# Define columns to drop (metadata)
drop_cols = ['Season', 'DayNum', 'WTeamID', 'LTeamID', 'WScore', 'LScore', 'WLoc', 'NumOT']
features = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
            'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg',
            'GamesPlayed_L', 'Wins_L', 'AvgScore_L', 'AvgFGP_L', 'Avg3PP_L', 'AvgFT_L',
            'AvgORP_L', 'AvgDRP_L', 'AvgASTTO_L', 'AvgNetRtg_L']

# Initial dataset (Team1 wins)
X = df[features]
y = np.ones(len(df), dtype=int)

# Invert data (Team2 wins)
df_inv = df.copy()
for col in features:
    if col.endswith('_L'):
        base_col = col[:-2]
        df_inv[col], df_inv[base_col] = df[base_col], df[col]

X_inv = df_inv[features]
y_inv = np.zeros(len(df_inv), dtype=int)

# Combine both datasets
X_train_full = pd.concat([X, X_inv], ignore_index=True)
y_train_full = np.concatenate([y, y_inv])

# ✅ FIX: Explicitly Replace infinite values
X_train_full.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check for NaNs or infinite values explicitly again
print("Infinite values present after replacement:", np.isinf(X_train_full).sum().sum())
print("NaNs before imputation:", X_train_full.isna().sum().sum())

# Impute missing values clearly
imputer = SimpleImputer(strategy='median')
X_train_full = pd.DataFrame(imputer.fit_transform(X_train_full), columns=features)

# Verify no missing or infinite values remain
assert not np.isinf(X_train_full.values).any(), "Infinite values remain!"
assert not np.isnan(X_train_full.values).any(), "NaN values remain after imputation."

# Train Random Forest clearly
rf_model = RandomForestClassifier(n_estimators=200, max_depth=7, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_model.fit(X_train_full, y_train_full)

print("✅ Model trained successfully without infinities or NaNs.")


Infinite values present after replacement: 0
NaNs before imputation: 174
✅ Model trained successfully without infinities or NaNs.


In [6]:
# Load submission_df
submission_df = pd.read_csv('data/SampleSubmissionStage2.csv')
submission_df[['Season', 'LowerID', 'HigherID']] = submission_df['ID'].str.split('_', expand=True).astype(int)

# Define team stat columns explicitly
team_stats_cols = ['GamesPlayed', 'Wins', 'AvgScore', 'AvgFGP', 'Avg3PP', 'AvgFT',
                   'AvgORP', 'AvgDRP', 'AvgASTTO', 'AvgNetRtg']

# Create a clear lookup DataFrame for team stats
winner_stats_df = df.set_index(['Season', 'WTeamID'])[team_stats_cols]
loser_stats_df = df.set_index(['Season', 'LTeamID'])[[col + '_L' for col in team_stats_cols]]
loser_stats_cols = {col + '_L': col for col in team_stats_cols}
loser_stats_df.rename(columns=loser_stats_cols, inplace=True)

# Combine winner and loser stats into one lookup DataFrame
all_team_stats_df = pd.concat([winner_stats_df, loser_stats_df])
all_team_stats_df = all_team_stats_df[~all_team_stats_df.index.duplicated(keep='first')]  # Remove duplicates clearly

# Function to safely get stats
def get_team_stats(season, team_id):
    if (season, team_id) in all_team_stats_df.index:
        stats = all_team_stats_df.loc[(season, team_id)]
        # ensure it returns a list from Series
        if isinstance(stats, pd.Series):
            return stats.tolist()
        else:  # If it's a DataFrame due to multiple matches, take the first row
            return stats.iloc[0].tolist()
    else:
        return X_train_full[team_stats_cols].median().tolist()

# Construct test set efficiently
X_test_rows = [
    get_team_stats(season, lower) + get_team_stats(season, higher)
    for season, lower, higher in submission_df[['Season', 'LowerID', 'HigherID']].itertuples(index=False)
]

X_test = pd.DataFrame(X_test_rows, columns=[f'{col}_1' for col in team_stats_cols] + [f'{col}_2' for col in team_stats_cols])

# Check the feature alignment clearly
print("✅ Features prepared for testing.")
print("Features used by model:", rf_model.feature_names_in_)
print("Test DataFrame columns:", X_test.columns.tolist())

missing_in_test = set(rf_model.feature_names_in_) - set(X_test.columns)
extra_in_test = set(X_test.columns) - set(rf_model.feature_names_in_)

print("Missing in test (should be empty):", missing_in_test)
print("Extra in test (should be empty):", extra_in_test)

# Confirm dimensions
print(X_test.shape) # Should clearly show (131407, 20)


✅ Features prepared for testing.
Features used by model: ['GamesPlayed' 'Wins' 'AvgScore' 'AvgFGP' 'Avg3PP' 'AvgFT' 'AvgORP'
 'AvgDRP' 'AvgASTTO' 'AvgNetRtg' 'GamesPlayed_L' 'Wins_L' 'AvgScore_L'
 'AvgFGP_L' 'Avg3PP_L' 'AvgFT_L' 'AvgORP_L' 'AvgDRP_L' 'AvgASTTO_L'
 'AvgNetRtg_L']
Test DataFrame columns: ['GamesPlayed_1', 'Wins_1', 'AvgScore_1', 'AvgFGP_1', 'Avg3PP_1', 'AvgFT_1', 'AvgORP_1', 'AvgDRP_1', 'AvgASTTO_1', 'AvgNetRtg_1', 'GamesPlayed_2', 'Wins_2', 'AvgScore_2', 'AvgFGP_2', 'Avg3PP_2', 'AvgFT_2', 'AvgORP_2', 'AvgDRP_2', 'AvgASTTO_2', 'AvgNetRtg_2']
Missing in test (should be empty): {'Wins', 'AvgDRP', 'AvgFT', 'AvgORP_L', 'Avg3PP_L', 'AvgNetRtg', 'Wins_L', 'AvgFGP', 'AvgFGP_L', 'Avg3PP', 'AvgScore', 'AvgFT_L', 'AvgASTTO_L', 'AvgORP', 'GamesPlayed', 'AvgASTTO', 'GamesPlayed_L', 'AvgDRP_L', 'AvgScore_L', 'AvgNetRtg_L'}
Extra in test (should be empty): {'AvgFGP_1', 'GamesPlayed_1', 'AvgNetRtg_2', 'AvgFGP_2', 'AvgDRP_2', 'AvgORP_1', 'AvgFT_2', 'AvgASTTO_2', 'Wins_1', 'Avg3PP_1', 'A

In [9]:
# Check initial infinite values and NaNs explicitly
print("Infinite values before replacement:", np.isinf(X_test.values).sum())
print("NaNs before replacement:", X_test.isna().sum().sum())

# Replace infinite values explicitly with NaN
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

# Verify intermediate NaNs after replacement
print("NaNs after replacing infinities:", X_test.isna().sum().sum())

# Explicitly rename columns BEFORE imputation to align with training data
rename_mapping = {
    'GamesPlayed_1': 'GamesPlayed',
    'Wins_1': 'Wins',
    'AvgScore_1': 'AvgScore',
    'AvgFGP_1': 'AvgFGP',
    'Avg3PP_1': 'Avg3PP',
    'AvgFT_1': 'AvgFT',
    'AvgORP_1': 'AvgORP',
    'AvgDRP_1': 'AvgDRP',
    'AvgASTTO_1': 'AvgASTTO',
    'AvgNetRtg_1': 'AvgNetRtg',
    'GamesPlayed_2': 'GamesPlayed_L',
    'Wins_2': 'Wins_L',
    'AvgScore_2': 'AvgScore_L',
    'AvgFGP_2': 'AvgFGP_L',
    'Avg3PP_2': 'Avg3PP_L',
    'AvgFT_2': 'AvgFT_L',
    'AvgORP_2': 'AvgORP_L',
    'AvgDRP_2': 'AvgDRP_L',
    'AvgASTTO_2': 'AvgASTTO_L',
    'AvgNetRtg_2': 'AvgNetRtg_L'
}
X_test.rename(columns=rename_mapping, inplace=True)

# Impute missing values explicitly using aligned columns with training median
for col in X_test.columns:
    median_val = X_train_full[col].median()
    X_test[col].fillna(median_val, inplace=True)

# Recheck explicitly to confirm no NaNs or infinities remain
assert np.isinf(X_test.values).sum() == 0, "Still contains infinities!"
assert X_test.isna().sum().sum() == 0, "Still contains NaNs!"

# Ensure data type consistency explicitly
X_test = X_test.astype(np.float64)

# Confirm feature alignment explicitly
assert set(X_test.columns) == set(rf_model.feature_names_in_), \
    f"Mismatch found: {set(rf_model.feature_names_in_) - set(X_test.columns)}"

# Generate predictions explicitly matching the model feature order
submission_df['Pred'] = rf_model.predict_proba(X_test[rf_model.feature_names_in_])[:, 1]

# Save predictions to CSV explicitly
submission_df[['ID', 'Pred']].to_csv('final_submission.csv', index=False)
print(submission_df[['ID', 'Pred']].shape)
print("✅ Final submission generated successfully: final_submission.csv")


Infinite values before replacement: 0
NaNs before replacement: 0
NaNs after replacing infinities: 0
(131407, 2)
✅ Final submission generated successfully: final_submission.csv
